# CAB AA/pp y=1 Crossing

This notebook reads the pipeline `eec.parquet` output and extracts the point where the CAB ratio crosses `y=1`.

The ratio is computed as `CAB_EEC(numerator) / CAB_EEC(denominator)`, using the same sample-pair logic as the plotting code: explicit `plot.ratio_pairs` from the YAML first, otherwise an inferred AA/pp-like pair such as `jewel_med/jewel_vac` or `PbPb/pp`.

The crossing is found by linear interpolation in `ln(R_L)`. Uncertainties are estimated by bootstrapping the ratio points with the propagated `cab_eec_err` uncertainty.

In [ ]:
from pathlib import Path
import math
import os

try:
    get_ipython().run_line_magic("matplotlib", "inline")
except NameError:
    os.environ.setdefault("MPLBACKEND", "Agg")
os.environ.setdefault("MPLCONFIGDIR", str(Path.cwd() / ".matplotlib"))
Path(os.environ["MPLCONFIGDIR"]).mkdir(parents=True, exist_ok=True)

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import yaml

try:
    from IPython.display import Image as ipy_image
    from IPython.display import display as ipy_display
except ImportError:
    ipy_image = None
    ipy_display = None

def in_ipython():
    try:
        return get_ipython() is not None
    except NameError:
        return False


IN_IPYTHON = in_ipython()
DISPLAY_INLINE_PLOTS = True


def display_figure(fig, path=None, close=True):
    if DISPLAY_INLINE_PLOTS and IN_IPYTHON:
        if ipy_display is not None:
            if path is not None and ipy_image is not None:
                ipy_display(ipy_image(filename=str(path)))
            else:
                ipy_display(fig)
                plt.show()
        if close:
            plt.close(fig)
        return
    if close:
        plt.close(fig)

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "config.yaml").exists() and (PROJECT_ROOT.parent / "config.yaml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

CONFIG_PATH = PROJECT_ROOT / "config.yaml"

# Override these only when you do not want the YAML/default ratio pair.
NUMERATOR_SAMPLE = None      # e.g. "jewel_med" or "PbPb"
DENOMINATOR_SAMPLE = None    # e.g. "jewel_vac" or "pp"
RATIO_LABEL = None           # e.g. "PbPb/pp"

# Optional filters. Leave as None to process all available groups.
SELECTIONS = None            # e.g. ["maxkt", "sd_z0p2"]
NORMALIZATIONS = None        # e.g. ["jet_pt2", "radiator_pt2", "radiator_scalar_sum_pt2"]
NORMALIZATION_ORDER = ["jet_pt2", "radiator_pt2", "radiator_scalar_sum_pt2", "per_prong"]
INCLUDE_LEGACY_PARENT_PT2 = False

# Crossing selection if a curve crosses y=1 more than once.
# Options: "first_in_lndr", "last_in_lndr", "closest_to_reference_rl".
# The reference policy avoids selecting tiny-R_L crossings from edge fluctuations.
CROSSING_POLICY = "closest_to_reference_rl"
REFERENCE_RL = 0.05

# Alternative smooth interpolation around the selected linear crossing.
POLY_DEGREE = 2
POLY_EXTRA_BINS_LEFT = 2
POLY_EXTRA_BINS_RIGHT = 3

BOOTSTRAP_REPLICAS = 2000
BOOTSTRAP_SEED = 12345
TARGET_RATIO = 1.0

with CONFIG_PATH.open("r", encoding="utf-8") as stream:
    config = yaml.safe_load(stream)

output_dir = Path(config["output_dir"])
if not output_dir.is_absolute():
    output_dir = PROJECT_ROOT / output_dir

plot_dir = Path(config.get("plot", {}).get("output_dir", output_dir / "plots"))
if not plot_dir.is_absolute():
    plot_dir = PROJECT_ROOT / plot_dir
plot_dir.mkdir(parents=True, exist_ok=True)

eec_path = output_dir / "eec.parquet"
summary_path = output_dir / "cab_ratio_y1_crossings.csv"
all_crossings_path = output_dir / "cab_ratio_y1_crossings_all.csv"

lndr_range = None
if "eec" in config and "lndr_min" in config["eec"] and "lndr_max" in config["eec"]:
    lndr_range = (float(config["eec"]["lndr_min"]), float(config["eec"]["lndr_max"]))

eec_path

## Load CAB Curves

In [ ]:
def normalize_sample_name(name):
    return "".join(c.lower() if c.isalnum() else "_" for c in str(name)).strip("_")


def find_sample(samples, candidates):
    normalized_candidates = tuple(normalize_sample_name(candidate) for candidate in candidates)
    for sample in samples:
        if normalize_sample_name(sample) in normalized_candidates:
            return sample
    for sample in samples:
        normalized = normalize_sample_name(sample)
        tokens = set(normalized.split("_"))
        if any(
            normalized.startswith(candidate + "_")
            or normalized.endswith("_" + candidate)
            or candidate in tokens
            for candidate in normalized_candidates
        ):
            return sample
    return None


def ratio_pairs_from_config_or_samples(config, samples):
    if NUMERATOR_SAMPLE is not None and DENOMINATOR_SAMPLE is not None:
        label = RATIO_LABEL or f"{NUMERATOR_SAMPLE}/{DENOMINATOR_SAMPLE}"
        return [{"numerator": NUMERATOR_SAMPLE, "denominator": DENOMINATOR_SAMPLE, "label": label}]

    configured = config.get("plot", {}).get("ratio_pairs")
    if configured:
        return [
            {
                "numerator": pair["numerator"],
                "denominator": pair["denominator"],
                "label": pair.get("label", f"{pair['numerator']}/{pair['denominator']}"),
            }
            for pair in configured
        ]

    pp = find_sample(samples, ("pp", "vac", "vacuum", "jewel_vac", "jewel_vacuum"))
    aa = find_sample(samples, ("pbpb", "pbpb_0_10", "pbpb_010", "aa", "med", "medium", "jewel_med", "jewel_medium"))
    if aa and pp:
        return [{"numerator": aa, "denominator": pp, "label": f"{aa}/{pp}"}]
    return []


columns = [
    "sample",
    "selection",
    "normalization",
    "bin_lo_lndr",
    "bin_hi_lndr",
    "bin_center_lndr",
    "cab_eec",
    "cab_eec_err",
    "n_jets",
]
eec_df = pq.read_table(eec_path, columns=columns).to_pandas()
samples = sorted(eec_df["sample"].unique())
ratio_pairs = ratio_pairs_from_config_or_samples(config, samples)

if SELECTIONS is None:
    selections = sorted(eec_df["selection"].unique())
else:
    selections = list(SELECTIONS)

available_normalizations = list(eec_df["normalization"].unique())
if NORMALIZATIONS is None:
    normalizations = [name for name in NORMALIZATION_ORDER if name in available_normalizations]
    extra_normalizations = sorted(
        name
        for name in available_normalizations
        if name not in NORMALIZATION_ORDER and (INCLUDE_LEGACY_PARENT_PT2 or name != "parent_pt2")
    )
    normalizations.extend(extra_normalizations)
else:
    normalizations = list(NORMALIZATIONS)

ratio_pairs, selections, normalizations

## Crossing Extraction

In [ ]:
def ratio_with_uncertainty(numerator, numerator_err, denominator, denominator_err):
    n = np.asarray(numerator, dtype=float)
    sn = np.asarray(numerator_err, dtype=float)
    d = np.asarray(denominator, dtype=float)
    sd = np.asarray(denominator_err, dtype=float)
    with np.errstate(divide="ignore", invalid="ignore"):
        ratio = np.where(d != 0, n / d, np.nan)
        rel_n = np.where(n != 0, sn / n, 0.0)
        rel_d = np.where(d != 0, sd / d, 0.0)
        ratio_err = np.abs(ratio) * np.sqrt(rel_n**2 + rel_d**2)
    ratio_err = np.where(np.isfinite(ratio), ratio_err, np.nan)
    return ratio, ratio_err


def cab_ratio_curve(df, numerator, denominator, selection, normalization):
    num = df[
        (df["sample"] == numerator)
        & (df["selection"] == selection)
        & (df["normalization"] == normalization)
    ].copy()
    den = df[
        (df["sample"] == denominator)
        & (df["selection"] == selection)
        & (df["normalization"] == normalization)
    ].copy()
    if num.empty or den.empty:
        return pd.DataFrame()

    keys = ["bin_lo_lndr", "bin_hi_lndr", "bin_center_lndr"]
    merged = num.merge(den, on=keys, suffixes=("_num", "_den"), validate="one_to_one")
    merged = merged.sort_values("bin_center_lndr").reset_index(drop=True)
    ratio, ratio_err = ratio_with_uncertainty(
        merged["cab_eec_num"].to_numpy(),
        merged["cab_eec_err_num"].to_numpy(),
        merged["cab_eec_den"].to_numpy(),
        merged["cab_eec_err_den"].to_numpy(),
    )
    merged["cab_ratio"] = ratio
    merged["cab_ratio_err"] = ratio_err
    merged["numerator"] = numerator
    merged["denominator"] = denominator
    merged["selection"] = selection
    merged["normalization"] = normalization
    return merged


def finite_xy(curve, y_values=None):
    x = curve["bin_center_lndr"].to_numpy(dtype=float)
    y = curve["cab_ratio"].to_numpy(dtype=float) if y_values is None else np.asarray(y_values, dtype=float)
    yerr = curve["cab_ratio_err"].to_numpy(dtype=float)
    mask = np.isfinite(x) & np.isfinite(y) & np.isfinite(yerr)
    return x[mask], y[mask], yerr[mask]


def crossing_candidates(x, y, target=TARGET_RATIO):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    candidates = []
    if len(x) == 0:
        return candidates

    z = y - float(target)
    for i in range(len(x) - 1):
        if not np.isfinite(z[i]) or not np.isfinite(z[i + 1]):
            continue
        if z[i] == 0.0:
            x_cross = float(x[i])
            fraction = 0.0
        elif z[i] * z[i + 1] > 0.0:
            continue
        elif z[i] == z[i + 1]:
            x_cross = float(0.5 * (x[i] + x[i + 1]))
            fraction = 0.5
        else:
            fraction = float((target - y[i]) / (y[i + 1] - y[i]))
            x_cross = float(x[i] + fraction * (x[i + 1] - x[i]))

        candidates.append(
            {
                "crossing_index": len(candidates),
                "left_bin_index": int(i),
                "right_bin_index": int(i + 1),
                "fraction": fraction,
                "crossing_lndr": x_cross,
                "crossing_rl": float(math.exp(x_cross)),
                "left_lndr": float(x[i]),
                "right_lndr": float(x[i + 1]),
                "left_ratio": float(y[i]),
                "right_ratio": float(y[i + 1]),
                "slope_dratio_dlndr": float((y[i + 1] - y[i]) / (x[i + 1] - x[i])),
            }
        )

    if len(x) and np.isfinite(z[-1]) and z[-1] == 0.0:
        candidates.append(
            {
                "crossing_index": len(candidates),
                "left_bin_index": int(len(x) - 1),
                "right_bin_index": int(len(x) - 1),
                "fraction": 0.0,
                "crossing_lndr": float(x[-1]),
                "crossing_rl": float(math.exp(x[-1])),
                "left_lndr": float(x[-1]),
                "right_lndr": float(x[-1]),
                "left_ratio": float(y[-1]),
                "right_ratio": float(y[-1]),
                "slope_dratio_dlndr": math.nan,
            }
        )

    return candidates


def select_crossing(candidates, policy=CROSSING_POLICY, reference_rl=REFERENCE_RL):
    if not candidates:
        return None
    if policy == "first_in_lndr":
        return min(candidates, key=lambda item: item["crossing_lndr"])
    if policy == "last_in_lndr":
        return max(candidates, key=lambda item: item["crossing_lndr"])
    if policy == "closest_to_reference_rl":
        reference_lndr = math.log(float(reference_rl))
        return min(candidates, key=lambda item: abs(item["crossing_lndr"] - reference_lndr))
    raise ValueError(f"Unknown CROSSING_POLICY: {policy}")


def selected_crossing_lndr(x, y, policy=CROSSING_POLICY):
    selected = select_crossing(crossing_candidates(x, y), policy=policy)
    return math.nan if selected is None else float(selected["crossing_lndr"])


def bootstrap_crossing(curve, rng):
    x, y, yerr = finite_xy(curve)
    if len(x) < 2:
        return np.asarray([], dtype=float)
    good_err = np.isfinite(yerr) & (yerr > 0)
    fallback = max(0.05 * float(np.nanmax(np.abs(y))), 1.0e-12) if np.any(np.isfinite(y)) else 1.0e-12
    sigma = np.where(good_err, yerr, fallback)
    crossings = []
    for _ in range(int(BOOTSTRAP_REPLICAS)):
        y_boot = rng.normal(y, sigma)
        x_cross = selected_crossing_lndr(x, y_boot)
        if np.isfinite(x_cross):
            crossings.append(x_cross)
    return np.asarray(crossings, dtype=float)


def polynomial_window_indices(n, left_idx, right_idx):
    lo = max(0, int(left_idx) - int(POLY_EXTRA_BINS_LEFT))
    hi = min(n, int(right_idx) + int(POLY_EXTRA_BINS_RIGHT) + 1)
    min_points = min(n, int(POLY_DEGREE) + 1)
    while hi - lo < min_points:
        if lo > 0:
            lo -= 1
        if hi - lo >= min_points:
            break
        if hi < n:
            hi += 1
        if lo == 0 and hi == n:
            break
    return np.arange(lo, hi, dtype=int)


def polynomial_crossing_from_linear(x, y, yerr, selected):
    x_linear = float(selected["crossing_lndr"])
    left_idx = int(selected["left_bin_index"])
    right_idx = int(selected["right_bin_index"])
    idx = polynomial_window_indices(len(x), left_idx, right_idx)
    degree = min(int(POLY_DEGREE), len(idx) - 1)
    if degree < 1 or len(idx) < degree + 1:
        return {"poly_status": "too_few_points", "poly_n_points": int(len(idx)), "poly_degree": int(degree)}

    x_fit = np.asarray(x[idx], dtype=float)
    y_fit = np.asarray(y[idx], dtype=float)
    yerr_fit = np.asarray(yerr[idx], dtype=float)
    mask = np.isfinite(x_fit) & np.isfinite(y_fit)
    if np.count_nonzero(mask) < degree + 1:
        return {"poly_status": "too_few_finite_points", "poly_n_points": int(np.count_nonzero(mask)), "poly_degree": int(degree)}

    x_fit = x_fit[mask]
    y_fit = y_fit[mask]
    yerr_fit = yerr_fit[mask]
    degree = min(degree, len(x_fit) - 1)
    dx = x_fit - x_linear
    good_err = np.isfinite(yerr_fit) & (yerr_fit > 0)
    weights = np.where(good_err, 1.0 / yerr_fit, 1.0)
    try:
        coeff = np.polyfit(dx, y_fit, degree, w=weights)
    except Exception as exc:
        return {"poly_status": f"fit_failed: {type(exc).__name__}", "poly_n_points": int(len(x_fit)), "poly_degree": int(degree)}

    root_coeff = coeff.copy()
    root_coeff[-1] -= float(TARGET_RATIO)
    roots = np.roots(root_coeff)
    real_roots = np.asarray([root.real for root in roots if abs(root.imag) < 1.0e-8], dtype=float)
    if len(real_roots) == 0:
        return {
            "poly_status": "no_real_root",
            "poly_n_points": int(len(x_fit)),
            "poly_degree": int(degree),
            "poly_window_lndr_min": float(np.min(x_fit)),
            "poly_window_lndr_max": float(np.max(x_fit)),
        }

    x_roots = x_linear + real_roots
    in_window = (x_roots >= np.min(x_fit)) & (x_roots <= np.max(x_fit))
    candidate_roots = x_roots[in_window] if np.any(in_window) else x_roots
    x_poly = float(candidate_roots[np.argmin(np.abs(candidate_roots - x_linear))])
    return {
        "poly_status": "ok",
        "poly_degree": int(degree),
        "poly_n_points": int(len(x_fit)),
        "poly_window_lndr_min": float(np.min(x_fit)),
        "poly_window_lndr_max": float(np.max(x_fit)),
        "poly_crossing_lndr": x_poly,
        "poly_crossing_rl": float(math.exp(x_poly)),
        "poly_delta_lndr_vs_linear": float(x_poly - x_linear),
        "poly_delta_rl_vs_linear": float(math.exp(x_poly) - math.exp(x_linear)),
        "poly_coefficients_high_to_low": ";".join(f"{value:.12g}" for value in coeff),
    }


def bootstrap_polynomial_crossing(curve, selected, rng):
    x, y, yerr = finite_xy(curve)
    if len(x) < 2:
        return np.asarray([], dtype=float)
    good_err = np.isfinite(yerr) & (yerr > 0)
    fallback = max(0.05 * float(np.nanmax(np.abs(y))), 1.0e-12) if np.any(np.isfinite(y)) else 1.0e-12
    sigma = np.where(good_err, yerr, fallback)
    crossings = []
    for _ in range(int(BOOTSTRAP_REPLICAS)):
        y_boot = rng.normal(y, sigma)
        result = polynomial_crossing_from_linear(x, y_boot, yerr, selected)
        if result.get("poly_status") == "ok" and np.isfinite(result.get("poly_crossing_lndr", math.nan)):
            crossings.append(float(result["poly_crossing_lndr"]))
    return np.asarray(crossings, dtype=float)


def crossing_summary_for_curve(curve, pair_label):
    x, y, yerr = finite_xy(curve)
    base = {
        "label": pair_label,
        "numerator": curve["numerator"].iloc[0] if not curve.empty else None,
        "denominator": curve["denominator"].iloc[0] if not curve.empty else None,
        "selection": curve["selection"].iloc[0] if not curve.empty else None,
        "normalization": curve["normalization"].iloc[0] if not curve.empty else None,
        "crossing_policy": CROSSING_POLICY,
        "target_ratio": float(TARGET_RATIO),
        "n_points": int(len(x)),
    }
    if len(x) < 2:
        return {**base, "status": "too_few_points", "n_crossings": 0}, []

    candidates = crossing_candidates(x, y)
    all_rows = [{**base, **candidate} for candidate in candidates]
    selected = select_crossing(candidates)
    if selected is None:
        nearest_idx = int(np.nanargmin(np.abs(y - TARGET_RATIO))) if len(y) else -1
        nearest = {
            "nearest_bin_lndr": float(x[nearest_idx]) if nearest_idx >= 0 else math.nan,
            "nearest_bin_rl": float(math.exp(x[nearest_idx])) if nearest_idx >= 0 else math.nan,
            "nearest_ratio": float(y[nearest_idx]) if nearest_idx >= 0 else math.nan,
        }
        return {**base, **nearest, "status": "no_crossing", "n_crossings": 0}, all_rows

    rng = np.random.default_rng(int(BOOTSTRAP_SEED))
    boot_lndr = bootstrap_crossing(curve, rng)
    boot_rl = np.exp(boot_lndr) if len(boot_lndr) else np.asarray([], dtype=float)
    uncertainty = {
        "crossing_lndr_err": float(np.std(boot_lndr, ddof=1)) if len(boot_lndr) > 1 else math.nan,
        "crossing_lndr_boot_p16": float(np.percentile(boot_lndr, 16.0)) if len(boot_lndr) else math.nan,
        "crossing_lndr_boot_p84": float(np.percentile(boot_lndr, 84.0)) if len(boot_lndr) else math.nan,
        "crossing_rl_err": float(np.std(boot_rl, ddof=1)) if len(boot_rl) > 1 else math.nan,
        "crossing_rl_boot_p16": float(np.percentile(boot_rl, 16.0)) if len(boot_rl) else math.nan,
        "crossing_rl_boot_p84": float(np.percentile(boot_rl, 84.0)) if len(boot_rl) else math.nan,
        "bootstrap_replicas_ok": int(len(boot_lndr)),
    }

    poly = polynomial_crossing_from_linear(x, y, yerr, selected)
    if poly.get("poly_status") == "ok":
        boot_poly_lndr = bootstrap_polynomial_crossing(curve, selected, rng)
        boot_poly_rl = np.exp(boot_poly_lndr) if len(boot_poly_lndr) else np.asarray([], dtype=float)
        poly.update(
            {
                "poly_crossing_lndr_err": float(np.std(boot_poly_lndr, ddof=1)) if len(boot_poly_lndr) > 1 else math.nan,
                "poly_crossing_lndr_boot_p16": float(np.percentile(boot_poly_lndr, 16.0)) if len(boot_poly_lndr) else math.nan,
                "poly_crossing_lndr_boot_p84": float(np.percentile(boot_poly_lndr, 84.0)) if len(boot_poly_lndr) else math.nan,
                "poly_crossing_rl_err": float(np.std(boot_poly_rl, ddof=1)) if len(boot_poly_rl) > 1 else math.nan,
                "poly_crossing_rl_boot_p16": float(np.percentile(boot_poly_rl, 16.0)) if len(boot_poly_rl) else math.nan,
                "poly_crossing_rl_boot_p84": float(np.percentile(boot_poly_rl, 84.0)) if len(boot_poly_rl) else math.nan,
                "poly_bootstrap_replicas_ok": int(len(boot_poly_lndr)),
            }
        )

    return {**base, **selected, **uncertainty, **poly, "status": "ok", "n_crossings": len(candidates)}, all_rows

In [ ]:
summary_rows = []
all_crossing_rows = []
curves = {}

if not ratio_pairs:
    raise RuntimeError("No ratio pair found. Set NUMERATOR_SAMPLE and DENOMINATOR_SAMPLE at the top of the notebook.")

for pair in ratio_pairs:
    numerator = pair["numerator"]
    denominator = pair["denominator"]
    label = pair.get("label", f"{numerator}/{denominator}")
    for selection in selections:
        for normalization in normalizations:
            curve = cab_ratio_curve(eec_df, numerator, denominator, selection, normalization)
            key = (label, selection, normalization)
            curves[key] = curve
            result, all_rows = crossing_summary_for_curve(curve, label) if not curve.empty else (
                {
                    "label": label,
                    "numerator": numerator,
                    "denominator": denominator,
                    "selection": selection,
                    "normalization": normalization,
                    "crossing_policy": CROSSING_POLICY,
                    "target_ratio": float(TARGET_RATIO),
                    "status": "missing_curve",
                    "n_points": 0,
                    "n_crossings": 0,
                },
                [],
            )
            summary_rows.append(result)
            all_crossing_rows.extend(all_rows)

summary_df = pd.DataFrame(summary_rows)
all_crossings_df = pd.DataFrame(all_crossing_rows)

summary_df.to_csv(summary_path, index=False)
all_crossings_df.to_csv(all_crossings_path, index=False)

summary_df.sort_values(["label", "normalization", "selection"])

The compact summary is written to `cab_ratio_y1_crossings.csv`. If a curve crosses the line more than once, all crossing candidates are also written to `cab_ratio_y1_crossings_all.csv`.

In [ ]:
summary_path, all_crossings_path

## Plot Checks

In [ ]:
def ratio_ylim_from_config(config):
    plot_cfg = config.get("plot", {})
    value = plot_cfg.get("ratio_cab_ylim", plot_cfg.get("ratio_ylim", [0.0, 4.0]))
    if value is None:
        return None
    return float(value[0]), float(value[1])


def add_linear_rl_axis(ax):
    lo, hi = ax.get_xlim()
    tick_values = [0.01, 0.02, 0.03, 0.05, 0.1, 0.2, 0.3, 0.4]
    ticks = [tick for tick in tick_values if lo <= np.log(tick) <= hi]
    if not ticks:
        return None
    secax = ax.secondary_xaxis(
        "top",
        functions=(lambda value: np.exp(value), lambda value: np.log(np.maximum(value, 1.0e-12))),
    )
    secax.set_xticks(ticks)
    secax.set_xticklabels([f"{tick:g}" for tick in ticks])
    secax.set_xlabel("R_L")
    return secax


PLOT_LABELS = None          # e.g. ["jewel_med/jewel_vac"]
PLOT_NORMALIZATIONS = None  # e.g. ["jet_pt2"]
PLOT_SELECTIONS = None      # e.g. ["maxkt", "sd_z0p2"]

plot_labels = sorted({key[0] for key in curves}) if PLOT_LABELS is None else list(PLOT_LABELS)
plot_normalizations = normalizations if PLOT_NORMALIZATIONS is None else list(PLOT_NORMALIZATIONS)
plot_selections = selections if PLOT_SELECTIONS is None else list(PLOT_SELECTIONS)
ylim = ratio_ylim_from_config(config)

saved_plots = []
for label in plot_labels:
    for normalization in plot_normalizations:
        fig, ax = plt.subplots(figsize=(7, 4))
        n_curves = 0
        for selection in plot_selections:
            curve = curves.get((label, selection, normalization), pd.DataFrame())
            if curve.empty:
                continue
            x, y, yerr = finite_xy(curve)
            if len(x) == 0:
                continue
            ax.step(x, y, where="mid", label=selection)
            ax.fill_between(x, y - yerr, y + yerr, alpha=0.12, step="mid")
            selected = summary_df[
                (summary_df["label"] == label)
                & (summary_df["selection"] == selection)
                & (summary_df["normalization"] == normalization)
                & (summary_df["status"] == "ok")
            ]
            if not selected.empty:
                row = selected.iloc[0]
                ax.axvline(row["crossing_lndr"], color="0.35", ls="--", lw=0.8, alpha=0.6)
                ax.plot(row["crossing_lndr"], TARGET_RATIO, marker="o", ms=4)
            n_curves += 1

        if n_curves == 0:
            plt.close(fig)
            continue
        ax.axhline(TARGET_RATIO, color="grey", ls=":", lw=1)
        ax.set_xlabel("ln(R_L)")
        ax.set_ylabel(label)
        ax.set_title(f"CAB y=1 crossing: {label}, {normalization}")
        if lndr_range is not None:
            ax.set_xlim(lndr_range)
        if ylim is not None:
            ax.set_ylim(ylim)
        ax.legend(fontsize=8)
        add_linear_rl_axis(ax)
        safe_label = "".join(c.lower() if c.isalnum() else "_" for c in label).strip("_")
        safe_norm = "".join(c.lower() if c.isalnum() else "_" for c in normalization).strip("_")
        out = plot_dir / f"cab_ratio_y1_crossing__{safe_label}__{safe_norm}.png"
        fig.savefig(out, dpi=150, bbox_inches="tight")
        display_figure(fig, out)
        saved_plots.append(out)

saved_plots

## Crossing Intercept Detail

The selected crossing is a local linear interpolation in `ln(R_L)` between the two bins that bracket `y=1`. The plotted line below is the same function used to locate the intercept:

`ratio(ln R_L) = 1 + slope * (ln R_L - crossing_lndr)`.

The line is drawn beyond the two bracketing bins only as a visual guide; the extracted crossing itself comes from the bracketing interval.

As an alternative, the notebook also fits a local polynomial around the selected linear intercept using `POLY_EXTRA_BINS_LEFT` and `POLY_EXTRA_BINS_RIGHT` additional neighboring bins. The polynomial crossing is stored in the `poly_*` columns and drawn as the green curve/marker.

In [ ]:
DETAIL_LABELS = None          # e.g. ["jewel_med/jewel_vac"]
DETAIL_NORMALIZATIONS = None  # e.g. ["jet_pt2"]
DETAIL_SELECTIONS = None      # e.g. ["maxkt", "sd_z0p2"]
DETAIL_LINE_EXTRA_BINS = 2
MAX_DETAIL_PLOTS = None       # e.g. 6; None shows all selected crossings


def row_allowed(row, labels, normalizations, selections):
    if labels is not None and row["label"] not in labels:
        return False
    if normalizations is not None and row["normalization"] not in normalizations:
        return False
    if selections is not None and row["selection"] not in selections:
        return False
    return True


def detail_line_range(x, left_idx, right_idx, extra_bins=DETAIL_LINE_EXTRA_BINS):
    lo_idx = max(0, int(left_idx) - int(extra_bins))
    hi_idx = min(len(x) - 1, int(right_idx) + int(extra_bins))
    return float(x[lo_idx]), float(x[hi_idx])


detail_rows = summary_df[summary_df["status"] == "ok"].copy()
detail_rows = detail_rows[
    detail_rows.apply(
        lambda row: row_allowed(row, DETAIL_LABELS, DETAIL_NORMALIZATIONS, DETAIL_SELECTIONS),
        axis=1,
    )
]
detail_rows = detail_rows.sort_values(["label", "normalization", "selection"])
if MAX_DETAIL_PLOTS is not None:
    detail_rows = detail_rows.head(int(MAX_DETAIL_PLOTS))

detail_plots = []
for _, row in detail_rows.iterrows():
    key = (row["label"], row["selection"], row["normalization"])
    curve = curves.get(key, pd.DataFrame())
    if curve.empty:
        continue
    x, y, yerr = finite_xy(curve)
    if len(x) < 2:
        continue

    x_cross = float(row["crossing_lndr"])
    rl_cross = float(row["crossing_rl"])
    slope = float(row["slope_dratio_dlndr"])
    left_idx = int(row["left_bin_index"])
    right_idx = int(row["right_bin_index"])
    if not np.isfinite(slope):
        continue

    x_lo, x_hi = detail_line_range(x, left_idx, right_idx)
    x_line = np.linspace(x_lo, x_hi, 100)
    y_line = TARGET_RATIO + slope * (x_line - x_cross)
    poly_status = row.get("poly_status", "")
    poly_x_line = None
    poly_y_line = None
    if poly_status == "ok" and isinstance(row.get("poly_coefficients_high_to_low"), str):
        coeff = np.asarray([float(value) for value in row["poly_coefficients_high_to_low"].split(";")], dtype=float)
        poly_lo = float(row["poly_window_lndr_min"])
        poly_hi = float(row["poly_window_lndr_max"])
        poly_x_line = np.linspace(poly_lo, poly_hi, 160)
        poly_y_line = np.polyval(coeff, poly_x_line - x_cross)

    fig, ax = plt.subplots(figsize=(7, 4))
    ax.errorbar(x, y, yerr=yerr, fmt="o", ms=3, lw=0.8, capsize=1.5, label="CAB ratio")
    ax.plot(x_line, y_line, color="tab:orange", lw=2.0, label="benchmark linear crossing")
    if poly_x_line is not None:
        ax.plot(poly_x_line, poly_y_line, color="tab:green", lw=2.0, label=f"polynomial degree {int(row['poly_degree'])}")
    ax.plot(x[[left_idx, right_idx]], y[[left_idx, right_idx]], "s", color="tab:red", ms=5, label="bracketing bins")
    ax.axhline(TARGET_RATIO, color="grey", ls=":", lw=1)
    ax.axvline(x_cross, color="black", ls="--", lw=1)
    ax.plot([x_cross], [TARGET_RATIO], marker="*", color="black", ms=10, label=f"linear R_L={rl_cross:.4g}")
    if poly_x_line is not None:
        ax.axvline(row["poly_crossing_lndr"], color="tab:green", ls="--", lw=1, alpha=0.8)
        ax.plot([row["poly_crossing_lndr"]], [TARGET_RATIO], marker="D", color="tab:green", ms=5, label=f"poly R_L={row['poly_crossing_rl']:.4g}")
    ax.set_xlabel("ln(R_L)")
    ax.set_ylabel(row["label"])
    ax.set_title(f"CAB y=1 intercept: {row['selection']}, {row['normalization']}")
    if lndr_range is not None:
        ax.set_xlim(lndr_range)
    if ylim is not None:
        ax.set_ylim(ylim)
    ax.legend(fontsize=8)
    add_linear_rl_axis(ax)

    safe_label = "".join(c.lower() if c.isalnum() else "_" for c in row["label"]).strip("_")
    safe_selection = "".join(c.lower() if c.isalnum() else "_" for c in row["selection"]).strip("_")
    safe_norm = "".join(c.lower() if c.isalnum() else "_" for c in row["normalization"]).strip("_")
    out = plot_dir / f"cab_ratio_y1_crossing_detail__{safe_label}__{safe_selection}__{safe_norm}.png"
    fig.savefig(out, dpi=150, bbox_inches="tight")
    detail_plots.append(out)
    display_figure(fig, out)

detail_plots

## Crossing Summary Plots

These summary plots use one extraction method at a time, controlled by `SUMMARY_METHOD`. Use `"linear"` for the two-bin benchmark crossing or `"poly"` for the local polynomial alternative.

In [ ]:
SUMMARY_METHOD = "linear"  # "linear" or "poly"
SUMMARY_LABELS = None       # e.g. ["jewel_med/jewel_vac"]
SUMMARY_NORMALIZATIONS = None


def method_columns(method):
    if method == "linear":
        return {
            "status": "status",
            "ok": "ok",
            "rl": "crossing_rl",
            "rl_err": "crossing_rl_err",
            "lndr": "crossing_lndr",
            "label": "linear benchmark",
        }
    if method == "poly":
        return {
            "status": "poly_status",
            "ok": "ok",
            "rl": "poly_crossing_rl",
            "rl_err": "poly_crossing_rl_err",
            "lndr": "poly_crossing_lndr",
            "label": f"polynomial degree {POLY_DEGREE}",
        }
    raise ValueError(f"Unknown SUMMARY_METHOD: {method}")


def zcut_from_selection(selection):
    text = str(selection)
    if not text.startswith("sd_") or "z" not in text:
        return math.nan
    token = text.split("z", 1)[1]
    token = token.replace("p", ".")
    try:
        return float(token)
    except ValueError:
        return math.nan


def filtered_summary_rows(method):
    fields = method_columns(method)
    rows = summary_df.copy()
    if SUMMARY_LABELS is not None:
        rows = rows[rows["label"].isin(SUMMARY_LABELS)]
    if SUMMARY_NORMALIZATIONS is not None:
        rows = rows[rows["normalization"].isin(SUMMARY_NORMALIZATIONS)]
    rows = rows[rows[fields["status"]] == fields["ok"]]
    rows = rows[np.isfinite(rows[fields["rl"]])]
    return rows


summary_fields = method_columns(SUMMARY_METHOD)
summary_rows_for_plot = filtered_summary_rows(SUMMARY_METHOD)
summary_plots = []

for label, label_rows in summary_rows_for_plot.groupby("label", sort=True):
    fig, ax = plt.subplots(figsize=(7, 4))
    n_curves = 0
    for normalization, norm_rows in label_rows.groupby("normalization", sort=True):
        rows = norm_rows.copy()
        rows["z_cut"] = rows["selection"].map(zcut_from_selection)
        rows = rows[np.isfinite(rows["z_cut"])].sort_values("z_cut")
        if rows.empty:
            continue
        ax.errorbar(
            rows["z_cut"],
            rows[summary_fields["rl"]],
            yerr=rows[summary_fields["rl_err"]],
            marker="o",
            capsize=2,
            lw=1.5,
            label=normalization,
        )
        n_curves += 1
    if n_curves == 0:
        plt.close(fig)
        continue
    ax.set_xlabel("Soft Drop z_cut")
    ax.set_ylabel("CAB ratio y=1 crossing R_L")
    ax.set_title(f"Soft Drop crossing vs z_cut: {label}, {summary_fields['label']}")
    ax.grid(True, alpha=0.25)
    ax.legend(fontsize=8)
    safe_label = "".join(c.lower() if c.isalnum() else "_" for c in label).strip("_")
    out = plot_dir / f"cab_ratio_y1_summary_softdrop__{safe_label}__{SUMMARY_METHOD}.png"
    fig.savefig(out, dpi=150, bbox_inches="tight")
    summary_plots.append(out)
    display_figure(fig, out)

summary_plots

## Crossing Shifts vs Selected Radiator pT

This section joins the CAB crossing summary to the selected-splitting table. The radiator pT is the stored `parent_pt`, i.e. the fjcontrib selected radiator `pair().perp()` scale with the scalar prong-sum fallback used during analysis.

For an AA/pp ratio, the x-axis can use the numerator radiator pT, denominator radiator pT, or their mean. The relative crossing shift is computed against the lowest available soft-drop `z_cut` for each `(ratio label, normalization)` by default. The stored `relative_delta_crossing_rl` column is dimensionless; the plot uses the corresponding percent column.

In [ ]:
RADIATOR_PT_SOURCE = "numerator"  # "numerator", "denominator", or "mean"
RELATIVE_REFERENCE = "lowest_zcut"  # currently supported: "lowest_zcut"


def weighted_mean_and_sem(values, weights=None):
    values = np.asarray(values, dtype=float)
    mask = np.isfinite(values)
    if weights is None:
        weights = np.ones_like(values)
    else:
        weights = np.asarray(weights, dtype=float)
        mask &= np.isfinite(weights) & (weights > 0)
    values = values[mask]
    weights = weights[mask]
    if len(values) == 0:
        return math.nan, math.nan, 0
    mean = float(np.average(values, weights=weights))
    variance = float(np.average((values - mean) ** 2, weights=weights))
    n_eff = float(np.sum(weights) ** 2 / np.sum(weights**2)) if np.sum(weights**2) > 0 else float(len(values))
    sem = math.sqrt(max(variance, 0.0) / max(n_eff, 1.0))
    return mean, sem, int(len(values))


def load_radiator_pt_summary(config):
    rows = []
    for sample in config["samples"]:
        sample_name = sample["name"]
        path = output_dir / f"{sample_name}__splittings.parquet"
        if not path.exists():
            continue
        table = pq.read_table(path, columns=["selection", "parent_pt", "pt_a", "pt_b", "jet_pt", "event_weight"])
        df = table.to_pandas()
        if df.empty:
            continue
        df["radiator_pt"] = df["parent_pt"].where(np.isfinite(df["parent_pt"]), df["pt_a"] + df["pt_b"])
        for selection, group in df.groupby("selection", sort=True):
            radiator_mean, radiator_sem, n_splittings = weighted_mean_and_sem(group["radiator_pt"], group["event_weight"])
            jet_mean, jet_sem, _ = weighted_mean_and_sem(group["jet_pt"], group["event_weight"])
            rows.append(
                {
                    "sample": sample_name,
                    "selection": selection,
                    "radiator_pt_mean": radiator_mean,
                    "radiator_pt_sem": radiator_sem,
                    "jet_pt_mean": jet_mean,
                    "jet_pt_sem": jet_sem,
                    "n_splittings": n_splittings,
                }
            )
    return pd.DataFrame(rows)


def add_pair_radiator_pt(summary_rows, radiator_summary):
    rows = summary_rows.copy()
    num = radiator_summary.rename(
        columns={
            "sample": "numerator",
            "radiator_pt_mean": "radiator_pt_mean_num",
            "radiator_pt_sem": "radiator_pt_sem_num",
            "jet_pt_mean": "jet_pt_mean_num",
            "jet_pt_sem": "jet_pt_sem_num",
            "n_splittings": "n_splittings_num",
        }
    )
    den = radiator_summary.rename(
        columns={
            "sample": "denominator",
            "radiator_pt_mean": "radiator_pt_mean_den",
            "radiator_pt_sem": "radiator_pt_sem_den",
            "jet_pt_mean": "jet_pt_mean_den",
            "jet_pt_sem": "jet_pt_sem_den",
            "n_splittings": "n_splittings_den",
        }
    )
    keep_num = [col for col in num.columns if col != "normalization"]
    rows = rows.merge(num[keep_num], on=["numerator", "selection"], how="left")
    keep_den = [col for col in den.columns if col != "normalization"]
    rows = rows.merge(den[keep_den], on=["denominator", "selection"], how="left")
    rows["radiator_pt_mean_pair"] = 0.5 * (rows["radiator_pt_mean_num"] + rows["radiator_pt_mean_den"])
    rows["radiator_pt_sem_pair"] = 0.5 * np.hypot(rows["radiator_pt_sem_num"], rows["radiator_pt_sem_den"])
    rows["delta_radiator_pt_mean_num_minus_den"] = rows["radiator_pt_mean_num"] - rows["radiator_pt_mean_den"]
    rows["delta_radiator_pt_sem_num_minus_den"] = np.hypot(rows["radiator_pt_sem_num"], rows["radiator_pt_sem_den"])
    return rows


def selected_radiator_pt_columns(source):
    if source == "numerator":
        return "radiator_pt_mean_num", "radiator_pt_sem_num", "AA radiator <pT>"
    if source == "denominator":
        return "radiator_pt_mean_den", "radiator_pt_sem_den", "pp radiator <pT>"
    if source == "mean":
        return "radiator_pt_mean_pair", "radiator_pt_sem_pair", "mean AA/pp radiator <pT>"
    raise ValueError(f"Unknown RADIATOR_PT_SOURCE: {source}")


radiator_pt_summary = load_radiator_pt_summary(config)
crossing_radiator_df = add_pair_radiator_pt(summary_rows_for_plot, radiator_pt_summary)
crossing_radiator_df["z_cut"] = crossing_radiator_df["selection"].map(zcut_from_selection)
crossing_radiator_df = crossing_radiator_df[np.isfinite(crossing_radiator_df["z_cut"])].copy()

pt_col, pt_err_col, pt_label = selected_radiator_pt_columns(RADIATOR_PT_SOURCE)
relative_rows = []
for (label, normalization), group in crossing_radiator_df.groupby(["label", "normalization"], sort=True):
    group = group.sort_values("z_cut")
    if group.empty:
        continue
    if RELATIVE_REFERENCE != "lowest_zcut":
        raise ValueError(f"Unknown RELATIVE_REFERENCE: {RELATIVE_REFERENCE}")
    reference = group.iloc[0]
    reference_rl = float(reference[summary_fields["rl"]])
    reference_rl_err = float(reference[summary_fields["rl_err"]])
    for _, row in group.iterrows():
        rl = float(row[summary_fields["rl"]])
        rl_err = float(row[summary_fields["rl_err"]])
        delta_rl = rl - reference_rl
        delta_rl_err = math.hypot(rl_err, reference_rl_err)
        with np.errstate(divide="ignore", invalid="ignore"):
            rel_delta = rl / reference_rl - 1.0 if reference_rl != 0 else math.nan
            rel_delta_err = abs(rl / reference_rl) * math.hypot(
                rl_err / rl if rl != 0 else 0.0,
                reference_rl_err / reference_rl if reference_rl != 0 else 0.0,
            )
        out = dict(row)
        out.update(
            {
                "reference_selection": reference["selection"],
                "reference_z_cut": float(reference["z_cut"]),
                "reference_crossing_rl": reference_rl,
                "delta_crossing_rl": delta_rl,
                "delta_crossing_rl_err": delta_rl_err,
                "relative_delta_crossing_rl": rel_delta,
                "relative_delta_crossing_rl_err": rel_delta_err,
                "relative_delta_crossing_rl_percent": 100.0 * rel_delta,
                "relative_delta_crossing_rl_percent_err": 100.0 * rel_delta_err,
            }
        )
        relative_rows.append(out)

crossing_radiator_summary_df = pd.DataFrame(relative_rows)
crossing_radiator_summary_path = output_dir / f"cab_ratio_y1_crossing_vs_radiator_pt__{SUMMARY_METHOD}.csv"
crossing_radiator_summary_df.to_csv(crossing_radiator_summary_path, index=False)

crossing_radiator_summary_df[
    [
        "label",
        "selection",
        "normalization",
        "z_cut",
        pt_col,
        "delta_radiator_pt_mean_num_minus_den",
        summary_fields["rl"],
        "relative_delta_crossing_rl_percent",
    ]
].sort_values(["label", "normalization", "z_cut"])

In [ ]:
radiator_relation_plots = []
for label, label_rows in crossing_radiator_summary_df.groupby("label", sort=True):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
    n_curves = 0
    for normalization, rows in label_rows.groupby("normalization", sort=True):
        rows = rows.sort_values(pt_col)
        if rows.empty:
            continue
        axes[0].errorbar(
            rows[pt_col],
            rows[summary_fields["rl"]],
            xerr=rows[pt_err_col],
            yerr=rows[summary_fields["rl_err"]],
            marker="o",
            capsize=2,
            lw=1.5,
            label=normalization,
        )
        axes[1].errorbar(
            rows[pt_col],
            rows["relative_delta_crossing_rl_percent"],
            xerr=rows[pt_err_col],
            yerr=rows["relative_delta_crossing_rl_percent_err"],
            marker="o",
            capsize=2,
            lw=1.5,
            label=normalization,
        )
        for _, row in rows.iterrows():
            axes[0].annotate(f"z={row['z_cut']:g}", (row[pt_col], row[summary_fields["rl"]]), fontsize=7, xytext=(4, 3), textcoords="offset points")
            axes[1].annotate(f"z={row['z_cut']:g}", (row[pt_col], row["relative_delta_crossing_rl_percent"]), fontsize=7, xytext=(4, 3), textcoords="offset points")
        n_curves += 1
    if n_curves == 0:
        plt.close(fig)
        continue
    axes[0].set_xlabel(f"{pt_label} [GeV]")
    axes[0].set_ylabel("CAB ratio y=1 crossing R_L")
    axes[0].set_title(f"Crossing vs radiator pT: {summary_fields['label']}")
    axes[1].axhline(0.0, color="grey", ls=":", lw=1)
    axes[1].set_xlabel(f"{pt_label} [GeV]")
    axes[1].set_ylabel("Relative crossing shift [%]")
    axes[1].set_title(f"Relative to lowest z_cut: {summary_fields['label']}")
    for ax in axes:
        ax.grid(True, alpha=0.25)
        ax.legend(fontsize=8)
    fig.suptitle(f"CAB AA/pp crossing and selected radiator pT: {label}")
    safe_label = "".join(c.lower() if c.isalnum() else "_" for c in label).strip("_")
    safe_source = "".join(c.lower() if c.isalnum() else "_" for c in RADIATOR_PT_SOURCE).strip("_")
    out = plot_dir / f"cab_ratio_y1_crossing_vs_radiator_pt__{safe_label}__{SUMMARY_METHOD}__{safe_source}.png"
    fig.savefig(out, dpi=150, bbox_inches="tight")
    radiator_relation_plots.append(out)
    display_figure(fig, out)

radiator_relation_plots, crossing_radiator_summary_path

In [ ]:
delta_radiator_relation_plots = []
delta_pt_col = "delta_radiator_pt_mean_num_minus_den"
delta_pt_err_col = "delta_radiator_pt_sem_num_minus_den"

for label, label_rows in crossing_radiator_summary_df.groupby("label", sort=True):
    fig, ax = plt.subplots(figsize=(7, 4))
    n_curves = 0
    for normalization, rows in label_rows.groupby("normalization", sort=True):
        rows = rows.sort_values(delta_pt_col)
        if rows.empty:
            continue
        ax.errorbar(
            rows[delta_pt_col],
            rows["relative_delta_crossing_rl_percent"],
            xerr=rows[delta_pt_err_col],
            yerr=rows["relative_delta_crossing_rl_percent_err"],
            marker="o",
            capsize=2,
            lw=1.5,
            label=normalization,
        )
        for _, row in rows.iterrows():
            ax.annotate(
                f"z={row['z_cut']:g}",
                (row[delta_pt_col], row["relative_delta_crossing_rl_percent"]),
                fontsize=7,
                xytext=(4, 3),
                textcoords="offset points",
            )
        n_curves += 1
    if n_curves == 0:
        plt.close(fig)
        continue
    ax.axhline(0.0, color="grey", ls=":", lw=1)
    ax.axvline(0.0, color="grey", ls=":", lw=1)
    ax.set_xlabel("<pT>_radiator(AA) - <pT>_radiator(pp) [GeV]")
    ax.set_ylabel("Relative crossing shift [%]")
    ax.set_title(f"Crossing shift vs AA-pp radiator pT difference: {label}, {summary_fields['label']}")
    ax.grid(True, alpha=0.25)
    ax.legend(fontsize=8)
    safe_label = "".join(c.lower() if c.isalnum() else "_" for c in label).strip("_")
    out = plot_dir / f"cab_ratio_y1_crossing_shift_vs_delta_radiator_pt__{safe_label}__{SUMMARY_METHOD}.png"
    fig.savefig(out, dpi=150, bbox_inches="tight")
    delta_radiator_relation_plots.append(out)
    display_figure(fig, out)

delta_radiator_relation_plots

In [ ]:
radiator_plane_plots = []

for label, label_rows in crossing_radiator_summary_df.groupby("label", sort=True):
    rows = label_rows.copy()
    if rows.empty:
        continue
    fig, ax = plt.subplots(figsize=(7, 5))
    values = rows["relative_delta_crossing_rl_percent"].to_numpy(dtype=float)
    finite_values = values[np.isfinite(values)]
    if len(finite_values):
        vmax = max(float(np.nanmax(np.abs(finite_values))), 1.0)
    else:
        vmax = 1.0
    markers = ["o", "s", "^", "D", "P", "X"]
    normalizations_for_plot = [norm for norm in normalizations if norm in set(rows["normalization"])]
    for marker, normalization in zip(markers, normalizations_for_plot):
        norm_rows = rows[rows["normalization"] == normalization].sort_values("z_cut")
        if norm_rows.empty:
            continue
        sc = ax.scatter(
            norm_rows["radiator_pt_mean_den"],
            norm_rows["radiator_pt_mean_num"],
            c=norm_rows["relative_delta_crossing_rl_percent"],
            cmap="coolwarm",
            vmin=-vmax,
            vmax=vmax,
            marker=marker,
            s=75,
            edgecolor="black",
            linewidth=0.5,
            label=normalization,
        )
        ax.errorbar(
            norm_rows["radiator_pt_mean_den"],
            norm_rows["radiator_pt_mean_num"],
            xerr=norm_rows["radiator_pt_sem_den"],
            yerr=norm_rows["radiator_pt_sem_num"],
            fmt="none",
            ecolor="0.35",
            elinewidth=0.8,
            capsize=1.5,
            alpha=0.6,
        )
        for _, row in norm_rows.iterrows():
            ax.annotate(
                f"z={row['z_cut']:g}",
                (row["radiator_pt_mean_den"], row["radiator_pt_mean_num"]),
                fontsize=7,
                xytext=(4, 3),
                textcoords="offset points",
            )
    lo = min(float(np.nanmin(rows["radiator_pt_mean_den"])), float(np.nanmin(rows["radiator_pt_mean_num"])))
    hi = max(float(np.nanmax(rows["radiator_pt_mean_den"])), float(np.nanmax(rows["radiator_pt_mean_num"])))
    pad = 0.05 * max(hi - lo, 1.0)
    ax.plot([lo - pad, hi + pad], [lo - pad, hi + pad], color="grey", ls=":", lw=1, label="AA = pp")
    ax.set_xlim(lo - pad, hi + pad)
    ax.set_ylim(lo - pad, hi + pad)
    ax.set_xlabel("<pT>_radiator(pp) [GeV]")
    ax.set_ylabel("<pT>_radiator(AA) [GeV]")
    ax.set_title(f"Crossing shift in radiator-pT plane: {label}, {summary_fields['label']}")
    ax.grid(True, alpha=0.25)
    ax.legend(fontsize=8, loc="best")
    cbar = fig.colorbar(sc, ax=ax)
    cbar.set_label("Relative crossing shift [%]")
    safe_label = "".join(c.lower() if c.isalnum() else "_" for c in label).strip("_")
    out = plot_dir / f"cab_ratio_y1_crossing_shift_radiator_pt_plane__{safe_label}__{SUMMARY_METHOD}.png"
    fig.savefig(out, dpi=150, bbox_inches="tight")
    radiator_plane_plots.append(out)
    display_figure(fig, out)

radiator_plane_plots

In [ ]:
radiator_plane_split_plots = []

for label, label_rows in crossing_radiator_summary_df.groupby("label", sort=True):
    rows = label_rows.copy()
    if rows.empty:
        continue
    normalizations_for_plot = [norm for norm in normalizations if norm in set(rows["normalization"])]
    if not normalizations_for_plot:
        continue

    values = rows["relative_delta_crossing_rl_percent"].to_numpy(dtype=float)
    finite_values = values[np.isfinite(values)]
    vmax = max(float(np.nanmax(np.abs(finite_values))), 1.0) if len(finite_values) else 1.0
    lo = min(float(np.nanmin(rows["radiator_pt_mean_den"])), float(np.nanmin(rows["radiator_pt_mean_num"])))
    hi = max(float(np.nanmax(rows["radiator_pt_mean_den"])), float(np.nanmax(rows["radiator_pt_mean_num"])))
    pad = 0.05 * max(hi - lo, 1.0)

    n_panels = len(normalizations_for_plot)
    ncols = 2 if n_panels > 1 else 1
    nrows = int(math.ceil(n_panels / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(6.2 * ncols, 4.8 * nrows), squeeze=False, sharex=True, sharey=True)
    sc = None
    for ax, normalization in zip(axes.ravel(), normalizations_for_plot):
        norm_rows = rows[rows["normalization"] == normalization].sort_values("z_cut")
        sc = ax.scatter(
            norm_rows["radiator_pt_mean_den"],
            norm_rows["radiator_pt_mean_num"],
            c=norm_rows["relative_delta_crossing_rl_percent"],
            cmap="coolwarm",
            vmin=-vmax,
            vmax=vmax,
            marker="o",
            s=85,
            edgecolor="black",
            linewidth=0.5,
        )
        ax.errorbar(
            norm_rows["radiator_pt_mean_den"],
            norm_rows["radiator_pt_mean_num"],
            xerr=norm_rows["radiator_pt_sem_den"],
            yerr=norm_rows["radiator_pt_sem_num"],
            fmt="none",
            ecolor="0.35",
            elinewidth=0.8,
            capsize=1.5,
            alpha=0.6,
        )
        for _, row in norm_rows.iterrows():
            ax.annotate(
                f"z={row['z_cut']:g}",
                (row["radiator_pt_mean_den"], row["radiator_pt_mean_num"]),
                fontsize=8,
                xytext=(4, 3),
                textcoords="offset points",
            )
        ax.plot([lo - pad, hi + pad], [lo - pad, hi + pad], color="grey", ls=":", lw=1)
        ax.set_xlim(lo - pad, hi + pad)
        ax.set_ylim(lo - pad, hi + pad)
        ax.set_title(normalization)
        ax.grid(True, alpha=0.25)

    for ax in axes.ravel()[n_panels:]:
        ax.set_visible(False)
    for ax in axes[-1, :]:
        if ax.get_visible():
            ax.set_xlabel("<pT>_radiator(pp) [GeV]")
    for ax in axes[:, 0]:
        if ax.get_visible():
            ax.set_ylabel("<pT>_radiator(AA) [GeV]")
    if sc is not None:
        cbar = fig.colorbar(sc, ax=[ax for ax in axes.ravel() if ax.get_visible()], shrink=0.85)
        cbar.set_label("Relative crossing shift [%]")
    fig.suptitle(f"CAB AA/pp crossing shift in radiator-pT plane by scale choice: {label}, {summary_fields['label']}")
    safe_label = "".join(c.lower() if c.isalnum() else "_" for c in label).strip("_")
    out = plot_dir / f"cab_ratio_y1_crossing_shift_radiator_pt_plane_split__{safe_label}__{SUMMARY_METHOD}.png"
    fig.savefig(out, dpi=150, bbox_inches="tight")
    radiator_plane_split_plots.append(out)
    display_figure(fig, out)

radiator_plane_split_plots

In [ ]:
MAXKT_SELECTION_NAME = "maxkt"

maxkt_summary_plots = []
for label, label_rows in summary_rows_for_plot[summary_rows_for_plot["selection"] == MAXKT_SELECTION_NAME].groupby("label", sort=True):
    rows = label_rows.copy()
    if SUMMARY_NORMALIZATIONS is None:
        norm_order = [norm for norm in normalizations if norm in set(rows["normalization"])]
    else:
        norm_order = [norm for norm in SUMMARY_NORMALIZATIONS if norm in set(rows["normalization"])]
    rows["normalization"] = pd.Categorical(rows["normalization"], categories=norm_order, ordered=True)
    rows = rows.sort_values("normalization")
    if rows.empty:
        continue

    x = np.arange(len(rows))
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.errorbar(
        x,
        rows[summary_fields["rl"]],
        yerr=rows[summary_fields["rl_err"]],
        marker="o",
        capsize=2,
        lw=1.5,
    )
    ax.set_xticks(x)
    ax.set_xticklabels([str(value) for value in rows["normalization"]], rotation=25, ha="right")
    ax.set_xlabel("Normalization")
    ax.set_ylabel("CAB ratio y=1 crossing R_L")
    ax.set_title(f"max-kT crossing by normalization: {label}, {summary_fields['label']}")
    ax.grid(True, axis="y", alpha=0.25)
    safe_label = "".join(c.lower() if c.isalnum() else "_" for c in label).strip("_")
    out = plot_dir / f"cab_ratio_y1_summary_maxkt__{safe_label}__{SUMMARY_METHOD}.png"
    fig.savefig(out, dpi=150, bbox_inches="tight")
    maxkt_summary_plots.append(out)
    display_figure(fig, out)

maxkt_summary_plots